* Run the first cell to clone the repo and enter the workdir. If you are running it locally, skip it
* For basic testing, run the models you would like the try at the first phase
* and optimize them at the third stage with hyperparameter search.
* If you want to check other coin performances, you can use the second stage but it's seperate from the main flow of the project.
* At the end, you can evaluate all trained models on trading profit as well as prediction logic with the interactive plots *(pay attention to clear train_sessions folder for evaluation to go smoothly).*

In [ ]:
# # Remove any existing cloned repo
# !rm -rf CrossCurrencyPrediction

# # Clone the repo
# !git clone https://github.com/mericdemirors/CrossCurrencyPrediction

# # Change directory to repo
# %cd CrossCurrencyPrediction/train_scripts

In [ ]:
import os
import gc
import random
import argparse

from train import train_with_args
from evaluate import create_evaluation_graphs
from profit_inference import profit_inference, compare_profits
from trading_agents import get_trading_agent_values

args_dict = dict(
    model_name="EncoderDecoderLSTM", # name of the model to use
    input_window=56, # = number of time-series data in the input
    output_window=8, # = number of time-series data in the output
    dropout=0.05, # dropout p for the models
    num_layers=2, # number of layers in the LSTM/GRU/Transformer models
    hidden_dim=32, # dimension size for the LSTM/GRU/Transformer models
    num_heads=4, # number of heads for the Multiheadattention/Transformer
    teacher_forcing_ratio=1, # teacher forcing start ratio (what percentage of the training samples will be predicted utilizing the real data for later values in the output_window)
    teacher_forcing_ratio_decrease=0.05, # decrease in the teacher forcing ratio after each epoch
    
    input_coins=["BTC", "ETH", "BNB", "XRP"],
    input_features=["open", "close", "low", "high"],
    output_coins=["BTC"],
    output_features=["open", "close", "low", "high"],
    dataset_name="IntervalLogReturnTransformLowHighRootCoinDataset", # name of the dataset to use
    # csv path to load for training dataset
    train_csv_path="/home/mericdemirors/Desktop/lecture slides/TUD_lectures/S2/deep_learning_architectures_and_methods/CrossCurrencyPrediction/data/BTC_ETH_BNB_XRP_6h_shared_train_toy.csv",
    # csv path to load for validation dataset
    val_csv_path="/home/mericdemirors/Desktop/lecture slides/TUD_lectures/S2/deep_learning_architectures_and_methods/CrossCurrencyPrediction/data/BTC_ETH_BNB_XRP_6h_shared_val_toy.csv",
    profit_inference_csv_path="/home/mericdemirors/Desktop/lecture slides/TUD_lectures/S2/deep_learning_architectures_and_methods/CrossCurrencyPrediction/data/BTC_ETH_BNB_XRP_6h_shared_val_toy.csv",
    bank_start=1000,
    augmentation_p=0.1, # probability of a sample being augmented (also the probability of each augmentation being applied to that sample)
    augmentation_noise_std=0.005, # std for the gaussian noise augmentation
    augmentation_constant_c=0, # min max limit for the constant to be added to all samples
    augmentation_scale_s=0, # min max limit for the scale to be multiplied with all samples
    transform_name="QuantileTransformer", # sklearn.preprocessing QuantileTransformer or PowerTransformer to use
    output_distribution="normal", # sklearn.preprocessing QuantileTransformer distribution type
    n_quantiles=500, # sklearn.preprocessing QuantileTransformer number of quantiles
    merge_count=2, # how many intervals to merge for the MergedIntervalsLogReturnTransformCoinDataset dataset
    price_loss_with_real=True, # what to base the predictions for the price_loss calculation at the LogReturnTransformPriceLossCoinDataset dataset
    price_loss_weight=0.002, # how to weight the price_loss calculation at the LogReturnTransformPriceLossCoinDataset dataset

    plot_weight_grad=0, # whether to plot weight and gradient plots at each epoch
    
    loss_name="MSE", # name of the loss to use
    additional_loss_fn_names=[],#["low_high_loss", "variance_reward"], # any additional loss/regularization to use
    additional_loss_weights=[],#[0,0], # and their coefficients
    l1_loss_weight=5e-06, # weight for the l1 regularization
    l2_loss_weight=5e-06, # weight for the l2 regularization
    
    batch_size=32, # batch size
    epochs=1, # number of epochs
    epoch_plot_step=-1, # whether to plot mid-epoch inference plots, if so: plots the inference plots after each 'epoch_plot_step'th batch pass
    
    optimizer_name="Adam", # name of the optimizer to use
    lr=0.001, # learning rate
    lr_patience=10, # patience for learning rate scheduler, how many epochs to wait before dropping the learning rate
    lr_decrease=0.95, # multiplicative decrease for the learning rate scheduler, what to multiply with to drop the learning rate
    early_stop_patience=10, # early stop after no improvement in validation score
)

# First phase trainings

In [ ]:
# ! TCN is deprecated because of tensor-dimension msimatch error
# args_dict["model_name"] = "TCN"
# args = argparse.Namespace(**args_dict)
# train_session_dir = train_with_args(args)
# create_evaluation_graphs(train_session_dir)
# toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
# regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
# compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderGRU"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderTransformer"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "CoinWiseCrossAttentionLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "FeatureWiseCrossAttentionLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

# Second phase trainings

In [ ]:
args_dict["output_coins"] = ["BTC"]

In [ ]:
args_dict["model_name"] = "EncoderDecoderGRU"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["output_coins"] = ["ETH"]

In [ ]:
args_dict["model_name"] = "EncoderDecoderGRU"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["output_coins"] = ["BNB"]

In [ ]:
args_dict["model_name"] = "EncoderDecoderGRU"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["output_coins"] = ["XRP"]

In [ ]:
args_dict["model_name"] = "EncoderDecoderGRU"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

In [ ]:
args_dict["model_name"] = "EncoderDecoderLSTM"
args = argparse.Namespace(**args_dict)
train_session_dir = train_with_args(args)
create_evaluation_graphs(train_session_dir)
toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])
compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])

# Third phase trainigs

In [ ]:
used_args = []

In [ ]:
regular_agent_values, _ = get_trading_agent_values(args_dict["profit_inference_csv_path"], args_dict["output_coins"][0], args_dict["bank_start"])

for _ in range(2):
    args_dict["input_window"] = random.choice([28, 42, 56])
    args_dict["output_window"] = random.choice([8, 16, 28])
    args_dict["dropout"] = random.choice([0, 0.05, 0.1, 0.2])
    args_dict["num_layers"] = random.choice([2, 4, 6, 8])
    args_dict["hidden_dim"] = random.choice([32, 64, 128])
    args_dict["augmentation_p"] = random.choice([0, 0.05, 0.1, 0.2,])
    args_dict["augmentation_noise_std"] = random.choice([0, 0.005, 0.02])
    args_dict["augmentation_constant_c"] = random.choice([0, 0.005, 0.02])
    args_dict["augmentation_scale_s"] = random.choice([0, 0.005, 0.01])
    args_dict["loss_name"] = random.choice(["MAE", "MSE", "SmoothL1"])
    args_dict["l1_loss_weight"] = random.choice([0, 5e-6])
    args_dict["l2_loss_weight"] = random.choice([0, 5e-6])
    if args_dict in [x[0] for x in used_args]:
        continue
    args = argparse.Namespace(**args_dict)
    
    train_session_dir = train_with_args(args)
    os.makedirs(os.path.join(train_session_dir, "evaluation_graphs"))
    
    # create_evaluation_graphs(train_session_dir)
    toast_bread_values, _, _ = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"])
    compare_profits(train_session_dir, regular_agent_values, toast_bread_values, args_dict["input_window"], args_dict["bank_start"])
    
    used_args.append((args_dict.copy(), toast_bread_values[-1]))
    gc.collect()

# Evaluating the models

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from trading_agents import *
from trading_agents import strategies

In [ ]:
df = pd.read_csv(args_dict["profit_inference_csv_path"], index_col="open_time")
df = df[[col for col in df.columns if "BTC" in col]].iloc[56:]
df.columns = [col.split("_")[1] for col in df.columns]

regular_agent_values = {}
buy_sells = {}
for agent_name, agent_func in strategies.items():
    value, buy_sell = agent_func(df.copy(), 1000)
    regular_agent_values[agent_name] = value
    buy_sells[agent_name] = len(buy_sell)

values_dict = regular_agent_values.copy()

In [ ]:
for e, train_session_dir in enumerate(os.listdir("train_sessions")):
    train_session_dir = os.path.join("train_sessions", train_session_dir)
    gc.collect()
    json_to_inference = os.path.join(train_session_dir,"args.json")
    with open(json_to_inference, 'r') as f:
        args_dict = json.load(f)

    toast_bread_values, buys, sells = profit_inference(train_session_dir, args_dict["profit_inference_csv_path"], args_dict["bank_start"], plot_interactive_plot=True)

    values_dict["toast_bread_" + str(e)] = toast_bread_values
    buy_sells["toast_bread_" + str(e)] = len(sorted([b[0] for b in buys] + [s[0] for s in sells]))
    orders_dict = buy_sells.copy()

In [ ]:
values_dict_correct_names = {}
orders_dict_correct_names = {}
for k,v in values_dict.items():
    if "agent" in k:
        values_dict_correct_names[k[:-6]] = values_dict[k]
        orders_dict_correct_names[k[:-6]] = orders_dict[k]
    else:
        values_dict_correct_names[k] = values_dict[k]
        orders_dict_correct_names[k] = orders_dict[k]
values_dict = values_dict_correct_names
orders_dict = orders_dict_correct_names

In [ ]:
plt.figure(figsize=(30, 15))

colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(regular_agent_values)))

max_value = args_dict["bank_start"]
for (k, v), c in zip(regular_agent_values.items(), colors):
    max_value = max(max(v), max_value)
    if k in ["volatility_agent", "low_high_slope_agent", "yesterday_trend_breaking_agent", "candlestick_pattern_agent", "buy_yesterdays_low_sell_yesterdays_high_agent"]:
        plt.plot(v, color=c, label=k, linestyle="dashed")
    else:
        plt.plot(v, color=c, label=k, linestyle="dotted")
    plt.text(len(v) - 1, v[-1], f" {k}", color=c, va="center", fontsize=15)

for y in range(args_dict["bank_start"], int(max_value), 1000):
    plt.axhline(y=y, color="gray", linestyle=":", linewidth=0.8)

for k,v in values_dict.items():
    if "toast_bread" in k:
        plt.plot(v, color="black", linestyle="solid", linewidth=2, label=k)
        plt.text(len(v) - 1, v[-1], " "+str(k), color="black", va="center", fontsize=15, fontweight="bold")

plt.legend()

plt.title(f'Trading Strategies Profit Comparison')
plt.xlabel("Time")
plt.ylabel("$ Value")
plt.show()

In [ ]:
# ---- Sort strategies by final value ----
sorted_strats = sorted(values_dict.keys(), key=lambda s: values_dict[s][-1], reverse=True)

# Assign distinct colors per strategy
base_colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(sorted_strats)))
color_map = {strat: base_colors[i] for i, strat in enumerate(sorted_strats)}

# Override toast_bread with black
for k,v in values_dict.items():
    if "toast_bread" in k:
        color_map[k] = "black"

In [ ]:
# ---- 1. Profit per trade ----
profits_per_trade = []
trades_list = []  # store the number of trades for annotation
for strat in sorted_strats:
    values = values_dict[strat]
    trades = orders_dict[strat] if orders_dict[strat] > 0 else 1
    trades_list.append(orders_dict[strat])  # store the actual trades
    profits_per_trade.append((values[-1] - values[0]) / trades)

bars = plt.bar(sorted_strats, profits_per_trade, color=[color_map[s] for s in sorted_strats])

# annotate bars with the trade counts
for bar, trades in zip(bars, trades_list):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(trades), ha="center", va="bottom", fontsize=9)

plt.title("Profit per Trade")
plt.ylabel("Profit per Trade")
plt.xticks(rotation=270)
plt.show()

In [ ]:
# ---- 2. Volatility (std of daily returns) ----
volatilities = []
for strat in sorted_strats:
    values = values_dict[strat]
    diffs = np.diff(values)
    volatilities.append(np.std(diffs))

plt.bar(sorted_strats, volatilities, color=[color_map[s] for s in sorted_strats])
plt.title("Standard deviation on interval values")
plt.ylabel("Standard deviation")
plt.xticks(rotation=270)
plt.show()

In [ ]:
# ---- 3. Up vs Down Days ----
up_counts, down_counts = [], []
for strat in sorted_strats:
    diffs = np.diff(values_dict[strat])
    up_counts.append(np.sum(diffs > 0) / len(diffs) * 100)
    down_counts.append(np.sum(diffs < 0) / len(diffs) * 100)

x = np.arange(len(sorted_strats))
plt.bar(x, up_counts, color=[color_map[s] for s in sorted_strats], label="Up days")
plt.bar(x, [-d for d in down_counts], color="lightgray", label="Down days")  # negative values
plt.axhline(0, color="black", linewidth=1)  # y=0 line
plt.xticks(x, sorted_strats, rotation=270)
plt.title("Up vs Down Intervals per Strategy")
plt.ylabel("Percentage of Intervals")
plt.legend()
plt.show()

In [ ]:
# ---- 4. Time below starting value ----
below_counts = []
for strat in sorted_strats:
    values = values_dict[strat]
    start_value = values[0]
    below_counts.append(np.sum(np.array(values) < start_value) / len(values) * 100)

plt.bar(sorted_strats, below_counts, color=[color_map[s] for s in sorted_strats])
plt.title("Time Spent Below Starting Value")
plt.ylabel("Percentage of Intervals")
plt.xticks(rotation=270)
plt.show()

# Zipping

In [ ]:
import shutil
shutil.make_archive("train_sessions_zipped", 'zip', "train_sessions")